## 환경설정 및 모델 설정

In [1]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm 
from dotenv import load_dotenv
import time
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# API 키 가져오기
load_dotenv()

model = ChatOpenAI(model="gpt-4o")  
embeddings = OpenAIEmbeddings()

## GQ 데이터 로드

In [13]:
# 1. 데이터 로드
file_path = "GotQuestions_raw_Ara_2025-05-01.xlsx"
df = pd.read_excel(file_path)

## 요약본 만들기(챗봇처럼)


In [15]:
def get_summary_response(question, original_answer):
    # 질문이나 답변이 너무 짧거나 비어있으면 API 호출 스킵
    if len(question.strip()) < 2 or len(original_answer.strip()) < 2:
        return ""

    system_message = """
    You are a wise and compassionate Christian counselor chatbot. 
    Your goal is to share the love of Jesus through gentle, conversational interactions.
    
    **Instructions:**
    1.  **Summarize:** The user provides a 'Question' and a 'Long Theological Answer'. You must rewrite the answer into a concise, conversational response.
    2.  **Tone:** Speak naturally like a caring friend or pastor, not like a textbook or a search engine. Be empathetic and warm.
    3.  **Length:** Keep it relatively short (2-4 sentences usually, unless the topic requires more nuance), suitable for a chat interface.
    4.  **Content:** Base your answer STRICTLY on the provided 'Long Theological Answer'. Do not invent new theology, but you can phrase it more simply.
    5.  **Style:** Avoid heavy theological jargon where possible. If the topic is sensitive, show understanding.
    """

    user_message = f"""
    **Question:** {question}
    
    **Long Theological Answer:** {original_answer}
    
    **Your Chatbot Response:**
    """
    try:
        response = model.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            temperature=0.5,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"  ⚠️ [Error] API 호출 실패 (질문: {question[:30]}...): {e}")
        return ""

# 4. 배치 처리 및 자동 저장 설정
print("\n🚀 전체 데이터셋 처리 및 자동 저장 시작...\n")

results = []
output_file = "GQ_Summarized_Full.xlsx"
backup_dir = "backup_logs" 

if not os.path.exists(backup_dir):
    os.makedirs(backup_dir)

# 데이터프레임 순회
for index, row in df.iterrows():
    q = row['Question_ENG']
    org_a = row['Answer_ENG']
    
    # ✅ 출력 시 안전하게 슬라이싱 (이미 위에서 str로 변환했으므로 안전함)
    print(f"[{index+1}/{len(df)}] 처리 중: {q[:40]}...", end="")
    
    # 요약 생성
    summary = get_summary_response(q, org_a)
    
    # 결과 저장
    results.append({
        'Question': q,
        'Original_Answer': org_a,
        'Chatbot_Summary': summary
    })
    
    print(" -> 완료")
    
    # -------------------------------------------------------
    # 100개마다 자동 저장
    # -------------------------------------------------------
    if (index + 1) % 100 == 0:
        temp_df = pd.DataFrame(results)
        backup_path = f"{backup_dir}/GQ_backup_{index+1}.xlsx"
        temp_df.to_excel(backup_path, index=False)
        print(f"\n💾 {index+1}개 처리 완료. 임시 저장됨: {backup_path}\n")
        
        time.sleep(1) 

# 5. 최종 결과 저장
final_df = pd.DataFrame(results)
final_df.to_excel(output_file, index=False)

print(f"\n🎉 모든 작업이 완료되었습니다! 최종 파일: '{output_file}'")

# 6. 실패한(빈 값) 개수 확인
failed_count = len(final_df[final_df['Chatbot_Summary'] == ""])
if failed_count > 0:
    print(f"⚠️ 주의: {failed_count}개의 질문에서 에러가 발생하거나 내용이 비어있어 요약되지 않았습니다.")


🚀 전체 데이터셋 처리 및 자동 저장 시작...

[1/2174] 처리 중: Got Eternal Life?... -> 완료
[2/2174] 처리 중: Got Forgiveness?... -> 완료
[3/2174] 처리 중: I am a Muslim. Why should I consider bec... -> 완료
[4/2174] 처리 중: What are the four spiritual laws?... -> 완료
[5/2174] 처리 중: How do I get right with God?... -> 완료
[6/2174] 처리 중: How can I know for sure that I will go t... -> 완료
[7/2174] 처리 중: Is Jesus the only way to heaven?... -> 완료
[8/2174] 처리 중: Is there life after death?... -> 완료
[9/2174] 처리 중: What does it mean to accept Jesus as you... -> 완료
[10/2174] 처리 중: What is the right religion for me?... -> 완료
[11/2174] 처리 중: What is the Romans Road to salvation?... -> 완료
[12/2174] 처리 중: What is the sinner’s prayer?... -> 완료
[13/2174] 처리 중: What is a Christian?... -> 완료
[14/2174] 처리 중: What does it mean to be a born again Chr... -> 완료
[15/2174] 처리 중: What is the way of salvation?... -> 완료
[16/2174] 처리 중: How can I become a Christian?... -> 완료
[17/2174] 처리 중: How can I become a child of God?... -> 완료
[18/2174] 처리 중: H

In [16]:
sum_df = pd.read_excel("GQ_Summarized_Full.xlsx")
print(sum_df)

                                               Question  \
0                                     Got Eternal Life?   
1                                      Got Forgiveness?   
2     I am a Muslim. Why should I consider becoming ...   
3                     What are the four spiritual laws?   
4                          How do I get right with God?   
...                                                 ...   
2169                                    What is Advent?   
2170                                  What is Carnival?   
2171              Should Christians celebrate Passover?   
2172                               What are Chreasters?   
2173  What are Septuagesima, Sexagesima, and Quinqua...   

                                        Original_Answer  \
0     The Bible presents a clear path to eternal lif...   
1     Acts 13:38\n declares, "Therefore, my brothers...   
2     People often follow the religion of their pare...   
3     The Four Spiritual Laws are a way of sharing t...

## 유사도 측정

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 요약 결과 파일 로드
input_file = "GQ_Summarized_Full.xlsx"
df = pd.read_excel(input_file)
print(f"파일 로드 성공: {len(df)}개 데이터")

# 2. 유사도 계산 함수
def calculate_similarity(row):
    doc1 = str(row['Original_Answer']).strip()
    doc2 = str(row['Chatbot_Summary']).strip()
    
    if not doc1 or not doc2:
        return 0.0
    
    # TF-IDF 벡터화 객체 생성
    tfidf_vectorizer = TfidfVectorizer()
    
    try:
        # 두 문장을 벡터로 변환
        tfidf_matrix = tfidf_vectorizer.fit_transform([doc1, doc2])
        
        # 코사인 유사도 계산 (0 ~ 1 사이 값)
        # 1에 가까울수록 문장에 사용된 단어 분포가 비슷함
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(similarity, 4) # 소수점 4자리까지 반올림
        
    except ValueError:
        # 문장이 너무 짧거나 단어가 없어서 벡터화가 불가능한 경우
        return 0.0

# 3. 전체 데이터에 적용
print("유사도 분석 중...")
# apply 함수로 각 행마다 유사도 계산 실행
df['Similarity_Score'] = df.apply(calculate_similarity, axis=1)

# 4. 결과 확인 및 저장
print("\n--- [유사도 분석 결과 미리보기] ---")
print(df[['Question', 'Chatbot_Summary', 'Similarity_Score']].head())

# 평균 유사도 출력
avg_sim = df['Similarity_Score'].mean()
print(f"\n>> 전체 평균 유사도: {avg_sim:.4f}")

# 유사도가 너무 낮은(예: 0.1 미만) 데이터 확인
low_sim_df = df[df['Similarity_Score'] < 0.1]
if not low_sim_df.empty:
    print(f">> 주의: 유사도가 0.1 미만인 데이터가 {len(low_sim_df)}개 있습니다.")
    print(low_sim_df[['Question', 'Chatbot_Summary']].head())

# 최종 파일 저장
output_file_with_score = "GQ_Summarized_with_Score.xlsx"
df.to_excel(output_file_with_score, index=False)
print(f"\n분석 완료! 결과가 '{output_file_with_score}'에 저장되었습니다.")

파일 로드 성공: 2174개 데이터
유사도 분석 중...

--- [유사도 분석 결과 미리보기] ---
                                            Question  \
0                                  Got Eternal Life?   
1                                   Got Forgiveness?   
2  I am a Muslim. Why should I consider becoming ...   
3                  What are the four spiritual laws?   
4                       How do I get right with God?   

                                     Chatbot_Summary  Similarity_Score  
0  Eternal life is a gift that comes from trustin...            0.5450  
1  Forgiveness is like wiping the slate clean; it...            0.6423  
2  It's understandable to question why you might ...            0.5511  
3  The Four Spiritual Laws are a simple way to un...            0.6009  
4  Getting right with God starts with realizing t...            0.6191  

>> 전체 평균 유사도: 0.5725

분석 완료! 결과가 'GQ_Summarized_with_Score.xlsx'에 저장되었습니다.


=> 0.57이면 나쁘지 않음! 전체 답변은 엄청 길지만 요약한 것인데, 그 중에서 사용한 단어가 비슷하다는 의미!

통과!!